# Nota

El notebook está diseñado para que corra completamente y muestre la tabla comparativa al final (antes de la prueba del modelo ganador)

* Ya que mi computador no cuenta con la capacidad de cómputo suficiente, se tomaron las siguientes medidas:
    * La tabla comparativa realizada con markdown se armó corriendo las pruebas de los modelos por separado.
    * Se comento la sección de Catboost por computo insuficiente; Al correr solo esa celda arroja RSME mayores a las del modelo elegido al final (~1700). 

## 

El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.
A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

# Inicialización

In [5]:
import warnings

import sys
import os
# Le dice python que busque liberrías ahí también
sys.path.append(os.path.join('../src'))
import funciones_personales as fp

import time
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# Preparación de datos

## Preprocesamiento y exploración de datos

In [6]:
df= pd.read_csv('../data/car_data.csv')
print(df.info())
#crea la lista de filas con valores nan deldataframe
los_nanes= fp.mostrar_nan(df, mostrar=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object
dtypes: int64(7), object(

### Búsqueda y tratamiento de valores nulos

**Se tienen un total de 108,555 filas con valores NaN de un total de 354,369 (~30.63 % del total de filas)**
* `VehicleType`: 37490
* `Gearbox`: 19833
* `Model`: 19705
* `FuelType`: 32895
* `NotRepaired`: 71154

¿ Qué tipo de datos tiene cada columna? y qué valores únicos tiene?

In [3]:
filas_c_nan= df[['VehicleType', 'Gearbox', 'Model', 'FuelType', 'NotRepaired']]

print(filas_c_nan.nunique())

for i in filas_c_nan.columns:
    print(f"Valores únicos en {i}: {filas_c_nan[i].unique()}")
    print('*'*50)

VehicleType      8
Gearbox          2
Model          250
FuelType         7
NotRepaired      2
dtype: int64
Valores únicos en VehicleType: [nan 'coupe' 'suv' 'small' 'sedan' 'convertible' 'bus' 'wagon' 'other']
**************************************************
Valores únicos en Gearbox: ['manual' 'auto' nan]
**************************************************
Valores únicos en Model: ['golf' nan 'grand' 'fabia' '3er' '2_reihe' 'other' 'c_max' '3_reihe'
 'passat' 'navara' 'ka' 'polo' 'twingo' 'a_klasse' 'scirocco' '5er'
 'meriva' 'arosa' 'c4' 'civic' 'transporter' 'punto' 'e_klasse' 'clio'
 'kadett' 'kangoo' 'corsa' 'one' 'fortwo' '1er' 'b_klasse' 'signum'
 'astra' 'a8' 'jetta' 'fiesta' 'c_klasse' 'micra' 'vito' 'sprinter' '156'
 'escort' 'forester' 'xc_reihe' 'scenic' 'a4' 'a1' 'insignia' 'combo'
 'focus' 'tt' 'a6' 'jazz' 'omega' 'slk' '7er' '80' '147' '100' 'z_reihe'
 'sportage' 'sorento' 'v40' 'ibiza' 'mustang' 'eos' 'touran' 'getz' 'a3'
 'almera' 'megane' 'lupo' 'r19' 'zafira' 'cadd

### Búsqueda y tratamiento de valores atípicos

In [4]:
# Muestra los datos estadísticos de cada columna del df
for i in df.columns:
    print(f"Valores estadísticos en {i}: {df[i].describe()}")
    print('*'*50)

Valores estadísticos en DateCrawled: count               354369
unique               15470
top       05/03/2016 14:25
freq                    66
Name: DateCrawled, dtype: object
**************************************************
Valores estadísticos en Price: count    354369.000000
mean       4416.656776
std        4514.158514
min           0.000000
25%        1050.000000
50%        2700.000000
75%        6400.000000
max       20000.000000
Name: Price, dtype: float64
**************************************************
Valores estadísticos en VehicleType: count     316879
unique         8
top        sedan
freq       91457
Name: VehicleType, dtype: object
**************************************************
Valores estadísticos en RegistrationYear: count    354369.000000
mean       2004.234448
std          90.227958
min        1000.000000
25%        1999.000000
50%        2003.000000
75%        2008.000000
max        9999.000000
Name: RegistrationYear, dtype: float64
***********************

In [5]:
# Se comentó ya que solo es una comprobación de datos**
'''
# Revisión de valores para los años de registro vehiculares
print(df['RegistrationYear'][(df['RegistrationYear'] < 1950)].value_counts().sort_index())
print('*'*50)
print(df['RegistrationYear'][(df['RegistrationYear'] > 2016)].value_counts().sort_index())
print(len(df[df['RegistrationMonth']==0]))
print(len(df['RegistrationMonth']))
'''

"\n# Revisión de valores para los años de registro vehiculares\nprint(df['RegistrationYear'][(df['RegistrationYear'] < 1950)].value_counts().sort_index())\nprint('*'*50)\nprint(df['RegistrationYear'][(df['RegistrationYear'] > 2016)].value_counts().sort_index())\nprint(len(df[df['RegistrationMonth']==0]))\nprint(len(df['RegistrationMonth']))\n"

**Valores Nulos:**
* Se decide rellenar los valores nulos con la palabra 'unknown'

**valores extraños en 'RegistrationYear'**
* Los primeros registros vehiculares datan de Francia en 1893.
* Existen 101 vehículos registrados en 1910. previo a este año hay registros de 1000 a 1800 (Deben modificarse).
* El año 2019 cuenta con 25 vehiculos registrados; depues los registros van hasta el año 2066 (deben modificarse).
    * Los años que se tomarán como verdaderos para este estudio son de 1910 a 2019.

**Creación de la variable `VehicleAge`**

La variable 'RegistrationYear' indica el año en que el vehículo fue matriculado. Sin embargo, para la predicción del precio resulta más informativo conocer la **antigüedad del vehículo**, ya que el valor de mercado suele disminuir conforme aumentan los años de uso.
* En el punto anterior se determinó que los años de matriculación válidos se encuentran entre 1910 y 2019.
    * Se decide crear una nueva variable denominada **VehicleAge**, calculada como: `VehicleAge = 2019 - RegistrationYear`

**Justificación**
* Representa de forma más directa la depreciación del vehículo.
* Reduce la escala de la variable, facilitando el entrenamiento de algunos modelos.
* Conserva toda la información relevante contenida en el año de matriculación.
    * Una vez creada la nueva característica, se eliminará del conjunto de datos para evitar redundancia entre ambas variables. 

**Valores = 0 en 'Power'**
* 40,225 de 354,369 (~11.35 del total)
    * Media: ~ 110.09
    * Mediana: ~ 105
    * El máximo (20,000: altísimo para auto de calle) y la desviación estandar (~ 189.85: elevada) apuntan a que los valores atípicos (outliers) están "jalando" hacia arriba la media.
* Se decide sustituir los valores 0 por el valor de la mediana (que es un poco más bajo que el de la media)

**Valores = 0 en 'RegistrationMonth'**
* 37,352 de 354,369 (~10.54% del total)
    * Al tratarse de columna númerica con datos inválidos se decide sustituir por NAN

**Valores atípicos en 'Price'**
* Según ivestigaciones un precio muy bajo de un auto que enciende y rueda es de 500 dólares / euros
* 36,054 de 354,369 (~ 10.17% del total) son menores de 500
* Se decide eliminar ese ~10.17% de datos quedando con alrededor de 312,000 para usarlos en el entrenamiento del modelo. (siguen siendo una buena cantidad)
* De tratar de rellenar estos datos puede "confundir" al modelo ya que está usando como target datos "inventados", no reales pudiendo sesgar las predicciones.

**Columnas no necesarias para el modelo**
* `DateCrawled`
* `DateCreated`
* `NumberOfPictures` 
    * Podría afectar pero todos sus valores son 0, por lo que no afecta en este caso
* `PostalCode`
    * Puede reflejar el poder adquisitivo de una región pero no directamente al precio del vehículo.
    * Se decide eliminar para simplificar el modelo (en caso de requerirse podría tomarse en cuenta para futuros modelos)
* `LastSeen`

In [6]:
# Rellena los nan con 'unknown' para cada columna con NAN
df = df.fillna({'VehicleType': 'unknown', 'Gearbox': 'unknown', 'Model': 'unknown', 'FuelType': 'unknown', 'NotRepaired': 'unknown'})

# Sustituye los valores de años fuera de rango por NAN
filtro_registro = ~df['RegistrationYear'].between(1910, 2019)
df.loc[filtro_registro, 'RegistrationYear'] = np.nan

# Cambia los valores de power de 0 por la mediana
df.loc[df['Power']== 0, 'Power'] = df['Power'].median()


# Sustituye los valores de 'RegistrationMonth' = 0 por NAN
filtro_mes= df['RegistrationMonth']== 0
df.loc[filtro_mes, 'RegistrationMonth'] = np.nan


# Elimina las filas cuyos datos en Price son menores de 500
df= df[df['Price'] >= 500]

# COMPROBACIONES
'''
# comprueba que los NAN se hayan sustituido por unknown
prueba_sin_nan= fp.mostrar_nan(df, mostrar=True)

# Comprueba que no queden valores para años fuera de rango
print(df['RegistrationYear'][(df['RegistrationYear'] < 1910)].value_counts().sort_index())
print('*'*50)
print(df['RegistrationYear'][(df['RegistrationYear'] > 2019)].value_counts().sort_index())

# Comprueba que no queden valores de 0 en power:
print(len(df[df['Power'] == 0]))

# Comprueba que no queden filas con RegistrationMonth= 0
print(len(df[df['RegistrationMonth'] == 0]))
'''


# Elimina las columnas que no se utilizarán en el modelo renombrando al df limpio como df_clean
df_clean = df.drop(['DateCrawled', 'DateCreated', 'PostalCode', 'LastSeen', 'NumberOfPictures'], axis=1)

# Se crea la variable de antigüedad del vehículo
df_clean['VehicleAge'] = 2019 - df_clean['RegistrationYear']

# Se elimina el año de matriculación ya innecesario
df_clean = df_clean.drop(columns='RegistrationYear')

#Comprpbación para df_clean
'''
print(df_clean.columns)
print(df_clean.info())
'''

'\nprint(df_clean.columns)\nprint(df_clean.info())\n'

## Estrategia de preprocesamiento para el entrenamiento de modelos

A partir del conjunto de datos limpio (`df_clean`) se utilizará un único proceso de división en entrenamiento, validación y prueba. Esto garantiza que todos los modelos sean evaluados sobre exactamente los mismos datos.

Cada familia de modelos requiere un preprocesamiento diferente, por lo que las transformaciones se se irán aplicando conforme sean necesarias.

### Modelos que utilizarán directamente `df_clean` con valores imputados

- **CatBoost**
- **LightGBM**

Ambos pueden trabajar con variables categóricas y valores faltantes mediante sus propios mecanismos internos, por lo que utilizarán los datos limpios sin codificación ni escalado.

### Modelos que requieren codificación One-Hot

- **Random Forest**
- **XGBoost**

Ambos modelos requieren que las variables categóricas sean transformadas mediante One-Hot Encoding (OHE).
* Para estos casos, únicamente las variables categóricas serán codificadas, mientras que las variables numéricas conservarán su escala original.

### Modelo que requiere escalado

La Regresión Lineal (modelo dummy) utilizará el mismo conjunto de datos codificado mediante OHE, además, las variables numéricas serán escaladas con `StandardScaler`.
* Este modelo se utilizará como modelo base (prueba de cordura) para comparar el desempeño de los modelos basados en árboles y de gradient boosting.

---

## Flujo general del proyecto
```
df_clean
      │
      ├── Train
      ├── Validation
      └── Test
             │
             ├────────────────────────────────────────────┐
             │                                            │
             │                                            │
      Pipeline Árboles                            Pipeline Regresión Lineal
             │                                            │
             │                                            │
      Imputación (fit en Train)                  Imputación (fit en Train)
             │                                            │
             │                                            │
      One-Hot Encoder                           One-Hot Encoder
      (handle_unknown='ignore')                 (drop='first')
             │                                            │
             │                                            │
      Random Forest                             StandardScaler
      XGBoost                                          │
             │                                         │
             ├──────────────► CatBoost                 │
             └──────────────► LightGBM                 │
                                                       │
                                                Regresión Lineal

```
**Nota**

La Regresión Lineal requiere un tratamiento diferente de las variables categóricas para evitar problemas de multicolinealidad perfecta derivados del One-Hot Encoding. Por ello se construyó un pipeline independiente utilizando `drop='first'`, mientras que los modelos basados en árboles utilizan todas las categorías, ya que no son sensibles a este problema.

### Prevención de Data Leakage

Para evitar fuga de información (data leakage o como me gusta llamarlo: "el modelo hace trampa"), todas las transformaciones del preprocesamiento seguirán el mismo principio:

* `fit()` únicamente sobre el conjunto de entrenamiento.
* `transform()` sobre los conjuntos de validación y prueba utilizando los parámetros aprendidos durante el entrenamiento.

Este procedimiento aplica para los tres procesos:
* Imputación de valores faltantes;
* Codificación One-Hot;
* Escalado de variables numéricas.

De esta manera, la información de los conjuntos de validación y prueba nunca participa durante el entrenamiento de los modelos.

In [7]:
# lista de las columnas categóricas y numéricas de los df
categoric_cols = [
    'VehicleType',
    'Gearbox',
    'Model',
    'FuelType',
    'Brand',
    'NotRepaired'
]

numeric_cols = [
    'VehicleAge',
    'Power',
    'Mileage',
    'RegistrationMonth'
]

## Separación de datos:

* df_clean cuenta con 318,315 filas y 11 columnas
    * Para aminorar tiempos de computo se decide dividir datos en train, validation y test. De esta manera quedarían aproximadamente:
        * ~222,820 para entrenar (datos suficientes)
        * ~47,747 para validar (ajuste de hiperparámetros)
        * ~47,747 para realizar evaluación final


In [8]:
# La división se utilizará para todos los modelos

# 1 Separación de target y features
X = df_clean.drop('Price', axis=1)
y = df_clean['Price']

# 2 División de datos (70/15/15) S
## Primera división: 85% (train+test) vs 15% (validation)
X_all, X_valid, y_all, y_valid = train_test_split(
    X, y, 
    test_size=0.15, 
    random_state=54321
)
## Segunda división: 70% train vs 15% test (del 85% restante)
### 15/85 ≈ 0.176 para obtener 15% del total
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, 
    test_size=0.176,  # 15/85 = 0.176
    random_state=54321
)

## Imputadores para datos con NaN


In [9]:
# Variables que utilizarán la mediana
median_cols = [
    'VehicleAge',
    'Power',
    'Mileage'
]

# Variable cuyo NaN significa "mes desconocido"
month_col = ['RegistrationMonth']

# Imputadores
median_imputer = SimpleImputer(strategy='median')

month_imputer = SimpleImputer(
    strategy='constant',
    fill_value=0
)

## Hacer fit (ajuste) solamente en Train:

In [10]:
# Crea X_train_median, X_train_month y finalmente X_train_num "juntando" los primeros dos
X_train_median = pd.DataFrame(
    median_imputer.fit_transform(X_train[median_cols]),
    columns=median_cols,
    index=X_train.index
)

X_train_month = pd.DataFrame(
    month_imputer.fit_transform(X_train[month_col]),
    columns=month_col,
    index=X_train.index
)

X_train_num = pd.concat(
    [X_train_median, X_train_month],
    axis=1
)

## Transformaciones de Validation y Test

In [11]:
# Validaciones
X_valid_median = pd.DataFrame(
    median_imputer.transform(X_valid[median_cols]),
    columns=median_cols,
    index=X_valid.index
)

X_valid_month = pd.DataFrame(
    month_imputer.transform(X_valid[month_col]),
    columns=month_col,
    index=X_valid.index
)

X_valid_num = pd.concat(
    [X_valid_median, X_valid_month],
    axis=1
)

# Pruebas (Test)
X_test_median = pd.DataFrame(
    median_imputer.transform(X_test[median_cols]),
    columns=median_cols,
    index=X_test.index
)

X_test_month = pd.DataFrame(
    month_imputer.transform(X_test[month_col]),
    columns=month_col,
    index=X_test.index
)

X_test_num = pd.concat(
    [X_test_median, X_test_month],
    axis=1
)


## Dataframes '..._clean' para CatBoost y LightBGM

In [12]:
# Se reconstruyen los DataFrames limpios
# (numéricas imputadas + categóricas originales)

X_train_clean = X_train.copy()
X_valid_clean = X_valid.copy()
X_test_clean = X_test.copy()

# Se reemplazan las variables numéricas originales
# por las ya imputadas

X_train_clean[median_cols] = X_train_num[median_cols]
X_train_clean[month_col] = X_train_num[month_col]

X_valid_clean[median_cols] = X_valid_num[median_cols]
X_valid_clean[month_col] = X_valid_num[month_col]

X_test_clean[median_cols] = X_test_num[median_cols]
X_test_clean[month_col] = X_test_num[month_col]



## One-Hot Encoding para RandomForest y XBoost

In [13]:
# sklearn 0.24.1 utiliza:
# sparse=False
# get_feature_names()


# Se crea el codificador:
ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse=False
)

# Se ajusta en train y se transforma en validations y test
X_train_cat = ohe.fit_transform(
    X_train[categoric_cols]
)

X_valid_cat = ohe.transform(
    X_valid[categoric_cols]
)

X_test_cat = ohe.transform(
    X_test[categoric_cols]
)

# Se crean los dataframes de categorías codificadas
ohe_cols = ohe.get_feature_names(categoric_cols)

X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=ohe_cols,
    index=X_train.index
)

X_valid_cat = pd.DataFrame(
    X_valid_cat,
    columns=ohe_cols,
    index=X_valid.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=ohe_cols,
    index=X_test.index
)

# Se juntan los df de valores imputados con los categóricos recien creados
X_train_hot = pd.concat(
    [X_train_num, X_train_cat],
    axis=1
)

X_valid_hot = pd.concat(
    [X_valid_num, X_valid_cat],
    axis=1
)

X_test_hot = pd.concat(
    [X_test_num, X_test_cat],
    axis=1
)

## Pipeline específico de Regresión Lineal

In [14]:
# ============================
# One-Hot Encoding exclusivo
# para Regresión Lineal
# ============================

## Obtiene los unos y ceros de los conjuntos de entrenamiento, validación y prueba tirando el primer elemento para evitar dependencia lineal

X_train_lr = pd.get_dummies(
    X_train_clean,
    columns=categoric_cols,
    drop_first=True
)

X_valid_lr = pd.get_dummies(
    X_valid_clean,
    columns=categoric_cols,
    drop_first=True
)

X_test_lr = pd.get_dummies(
    X_test_clean,
    columns=categoric_cols,
    drop_first=True
)

# Igualar columnas

X_valid_lr = X_valid_lr.reindex(
    columns=X_train_lr.columns,
    fill_value=0
)

X_test_lr = X_test_lr.reindex(
    columns=X_train_lr.columns,
    fill_value=0
)

## Escalamiento de los datos con fit solo en train y transform en valid y test
scaler_lr = StandardScaler()

X_train_lr_scaled = pd.DataFrame(
    scaler_lr.fit_transform(X_train_lr),
    columns=X_train_lr.columns,
    index=X_train_lr.index
)

X_valid_lr_scaled = pd.DataFrame(
    scaler_lr.transform(X_valid_lr),
    columns=X_train_lr.columns,
    index=X_valid_lr.index
)

X_test_lr_scaled = pd.DataFrame(
    scaler_lr.transform(X_test_lr),
    columns=X_train_lr.columns,
    index=X_test_lr.index
)


### Verificación de los conjuntos de datos generados

In [15]:
print(X_train_clean.isna().sum().sum())
print(X_valid_clean.isna().sum().sum())
print(X_test_clean.isna().sum().sum())

print()

print("X_train_hot :", X_train_hot.shape)
print("X_valid_hot :", X_valid_hot.shape)
print("X_test_hot  :", X_test_hot.shape)

print()

print("X_train_lr_scaled :", X_train_lr_scaled.shape)
print("X_valid_lr_scaled :", X_valid_lr_scaled.shape)
print("X_test_lr_scaled  :", X_test_lr_scaled.shape)

print()

print("NaN en X_train_hot:", X_train_hot.isna().sum().sum())
print("NaN en X_train_lr_scaled:", X_train_lr_scaled.isna().sum().sum())

0
0
0

X_train_hot : (222947, 317)
X_valid_hot : (47748, 317)
X_test_hot  : (47620, 317)

X_train_lr_scaled : (222947, 311)
X_valid_lr_scaled : (47748, 311)
X_test_lr_scaled  : (47620, 311)

NaN en X_train_hot: 0
NaN en X_train_lr_scaled: 0


# Fin del preprocesamiento; Inicio del "Machín" learning

## Entrenamiento y duración de entrenamiento de modelos
* Se decide crear una función (obviamente puede ser ubicada en `funciones_personales.py` en la carpeta `src`) ya que para cada modelo se realizarán los mismos pasos:
    1. Entrenar
    2. Cronometrar
    3. Predecir
    4. Cronometrar
    5. Calcular RMSE
    6. Guardar resultados

## Modelo dummy (regresión lineal)
**Este proceso lo aguanta cualquier computadora, no hay peligro de quedarse pegado 1 día**

In [16]:
modelo_lr = LinearRegression()

resultados_lr, modelo_lr = fp.evaluar_modelo(
    modelo=modelo_lr,
    X_train=X_train_lr_scaled,
    y_train=y_train,
    X_valid=X_valid_lr_scaled,
    y_valid=y_valid,
    nombre_modelo="Regresión Lineal"
)

resultados_lr

,Modelo,Parámetros,RMSE,Tiempo entrenamiento (s),Tiempo predicción (s)
0,Regresión Lineal,Default,2852.668244,8.112455,0.01284


## Random Forest
**Este proceso (primera opción comentada de la celda) no lo aguanta ni mi compu y tampoco el servidor de TripleTen, córralo bajo su propio riesgo**

In [17]:
# Crea la lista de parámetros
# CON LOS HIPERPARÁMETROS PROPUESTOS INICIALMENTE NI LA PLATAFORMA NI MI COOMPU Y PROBABLEMENTE LA TUYA TAMPOCO PUEDA COMPUTAR ESTO; POR ESO SE DEJARÁ COMENTADO
'''
parametros_rf = [

    {
        'n_estimators':100,
        'max_depth':10,
        'random_state':54321,
        'n_jobs':-1
    },

    {
        'n_estimators':200,
        'max_depth':15,
        'random_state':54321,
        'n_jobs':-1
    },

    {
        'n_estimators':300,
        'max_depth':20,
        'random_state':54321,
        'n_jobs':-1
    }

]

'''

# Opción 2 reducir los estimators y profundidades:
parametros_rf = [
    {
        "n_estimators": 50,
        "max_depth": 10,
        "random_state": 54321,
        "n_jobs": -1
    },
    {
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 54321,
        "n_jobs": -1
    },
    {
        "n_estimators": 100,
        "max_depth": 15,
        "random_state": 54321,
        "n_jobs": -1
    }
]

# Entrena al modelo
resultados_rf, mejor_rf = fp.probar_hiperparametros(
    modelo_base=RandomForestRegressor,
    lista_parametros=parametros_rf,
    X_train=X_train_hot,
    y_train=y_train,
    X_valid=X_valid_hot,
    y_valid=y_valid,
    nombre_modelo="Random Forest"
)

resultados_rf


,Modelo,Parámetros,RMSE,Tiempo entrenamiento (s),Tiempo predicción (s)
0,Random Forest,"n_estimators=100, max_depth=15, random_state=5...",1746.207757,123.188662,0.336256
1,Random Forest,"n_estimators=100, max_depth=10, random_state=5...",1969.912550,100.879214,0.184651
2,Random Forest,"n_estimators=50, max_depth=10, random_state=54...",1972.165634,52.960126,0.171601


# LightBGM

In [18]:
# transforma las columnas categóricas a `category`
for col in categoric_cols:
    X_train_clean[col] = X_train_clean[col].astype("category")
    X_valid_clean[col] = X_valid_clean[col].astype("category")

# Lista de diferentes parámetros
parametros_lgbm = [

    {
        'n_estimators':200,
        'learning_rate':0.1,
        'random_state':54321
    },

    {
        'n_estimators':400,
        'learning_rate':0.05,
        'random_state':54321
    }

]

# Entrenamiento
resultados_lgbm, mejor_lgbm = fp.probar_hiperparametros(
    modelo_base=LGBMRegressor,
    lista_parametros=parametros_lgbm,
    X_train=X_train_clean,
    y_train=y_train,
    X_valid=X_valid_clean,
    y_valid=y_valid,
    nombre_modelo="LightGBM"
)

resultados_lgbm

,Modelo,Parámetros,RMSE,Tiempo entrenamiento (s),Tiempo predicción (s)
0,LightGBM,"n_estimators=400, learning_rate=0.05, random_s...",1642.758100,7.190288,0.719389
1,LightGBM,"n_estimators=200, learning_rate=0.1, random_st...",1647.501131,3.467740,0.314992


# CatBoost

CatBoost permite trabajar directamente con variables categóricas, por lo que se utilizarán los DataFrames `X_train_clean` y `X_valid_clean`, manteniendo las variables categóricas en su formato original.

Se probarán diferentes combinaciones de hiperparámetros para comparar el desempeño del modelo en términos de RECM (RMSE), tiempo de entrenamiento y tiempo de predicción.

In [ ]:
'''
# Definimos las columnas categóricas
cat_features = [
    X_train_clean.columns.get_loc(col)
    for col in categoric_cols
]

# Lista de hiperparámetros


parametros_catboost = [
    {
        'iterations': 200,
        'learning_rate': 0.1,
        'depth': 6,
        'random_seed': 54321,
        'verbose': False
    },
    {
        'iterations': 400,
        'learning_rate': 0.05,
        'depth': 6,
        'random_seed': 54321,
        'verbose': False
    },
    {
        'iterations': 300,
        'learning_rate': 0.05,
        'depth': 8,
        'random_seed': 54321,
        'verbose': False
    }
]

parametros_catboost = [
    {
        'iterations': 100,
        'learning_rate': 0.1,
        'depth': 5,
        'random_seed': 54321,
        'verbose': False
    },
    {
        'iterations': 150,
        'learning_rate': 0.1,
        'depth': 6,
        'random_seed': 54321,
        'verbose': False
    },
    {
        'iterations': 200,
        'learning_rate': 0.05,
        'depth': 6,
        'random_seed': 54321,
        'verbose': False
    }
]

# Ejecución de Catboost
# Resultados temporales de CatBoost
lista_resultados_catboost = []

for parametros in parametros_catboost:

    modelo_catboost = CatBoostRegressor(
        **parametros
    )

    resultado, modelo_catboost = fp.evaluar_modelo(
        modelo=modelo_catboost,
        X_train=X_train_clean,
        y_train=y_train,
        X_valid=X_valid_clean,
        y_valid=y_valid,
        nombre_modelo="CatBoost",
        parametros=parametros,
        cat_features=cat_features
    )

    lista_resultados_catboost.append(resultado)


# Tabla final de resultados de CatBoost
resultados_catboost = pd.concat(
    lista_resultados_catboost,
    ignore_index=True
)

resultados_catboost

'''

# XGBoost

In [17]:
# Lista de hiperparámetros
parametros_xgb = [

    {
        'n_estimators':200,
        'learning_rate':0.1,
        'max_depth':6,
        'random_state':54321,
        'objective':'reg:squarederror'
    },

    {
        'n_estimators':400,
        'learning_rate':0.05,
        'max_depth':8,
        'random_state':54321,
        'objective':'reg:squarederror'
    }

]

# Entrenamiento
resultados_xgb, mejor_xgb = fp.probar_hiperparametros(
    modelo_base=XGBRegressor,
    lista_parametros=parametros_xgb,
    X_train=X_train_hot,
    y_train=y_train,
    X_valid=X_valid_hot,
    y_valid=y_valid,
    nombre_modelo="XGBoost"
)

resultados_xgb

,Modelo,Parámetros,RMSE,Tiempo entrenamiento (s),Tiempo predicción (s)
0,XGBoost,"n_estimators=400, learning_rate=0.05, max_dept...",1655.913399,911.209117,0.572720
1,XGBoost,"n_estimators=200, learning_rate=0.1, max_depth...",1715.734997,345.593016,0.340382


# Tabla comparativa

In [ ]:
resultados_finales = pd.concat(
    [
        resultados_lgbm,
        resultados_xgb,
        resultados_rf,
        resultados_lr
        #resultados_catboost
    ],
    ignore_index=True
)

resultados_finales = resultados_finales.sort_values(
    'RMSE'
).reset_index(drop=True)

resultados_finales

# Comparación de modelos

| Modelo           | Parámetros                                                        | RMSE       | Tiempo entrenamiento (s) | Tiempo predicción (s) |
|------------------|-------------------------------------------------------------------|------------|---------------------------|------------------------|
| LightGBM         | `n_estimators=400, learning_rate=0.05, random_state=...`         | 1642.758100 | 10.214282                 | 0.706271               |
| LightGBM         | `n_estimators=200, learning_rate=0.1, random_state=...`          | 1647.501131 | 6.539336                  | 0.388630               |
| XGBoost          | `n_estimators=400, learning_rate=0.05, max_depth=...`            | 1655.913399 | 946.358664                | 0.532078               |
| XGBoost          | `n_estimators=200, learning_rate=0.1, max_depth=...`             | 1715.734997 | 354.315043                | 0.323467               |
| Random Forest    | `n_estimators=100, max_depth=15, random_state=5...`              | 1746.207757 | 127.775747                | 0.387703               |
| Random Forest    | `n_estimators=100, max_depth=10, random_state=5...`              | 1969.912550 | 100.711246                | 0.205978               |
| Random Forest    | `n_estimators=50, max_depth=10, random_state=54...`              | 1972.165634 | 50.945966                 | 0.124617               |
| Regresión Lineal | `Default`                                                         | 2852.668244 | 7.856996                  | 0.015306               |

# Prueba del modelo ganador

## Evaluación final del modelo seleccionado

Después de comparar los modelos utilizando el conjunto de entrenamiento y validación, se seleccionó **LightGBM** como el modelo con mejor desempeño.

La configuración seleccionada fue:

- `n_estimators = 400`
- `learning_rate = 0.05`
- `random_state = 54321`

El modelo se evaluará finalmente sobre el conjunto de prueba (`test`), que no fue utilizado durante la selección de hiperparámetros.

Se calcularán:
- RECM (RMSE)
- Tiempo de predicción

In [23]:
# Aseguramos que las categorías de test sean exactamente
# las mismas que las utilizadas durante el entrenamiento

for col in categoric_cols:
    X_test_clean[col] = X_test_clean[col].astype("category")
    
    X_test_clean[col] = X_test_clean[col].cat.set_categories(
        X_train_clean[col].cat.categories
    )
    
# prueba que efectivamente sean iguales
for col in categoric_cols:
    print(
        col,
        X_train_clean[col].cat.categories.equals(
            X_test_clean[col].cat.categories
        )
    )

VehicleType True
Gearbox True
Model True
FuelType True
Brand True
NotRepaired True


In [24]:
## Predicciones del ganador

inicio_pred = time.perf_counter()

predicciones_test = mejor_lgbm.predict(X_test_clean)

tiempo_pred_test = time.perf_counter() - inicio_pred

rmse_test = mean_squared_error(
    y_test,
    predicciones_test,
    squared=False
)

print("RMSE test:", rmse_test)
print("Tiempo de predicción:", tiempo_pred_test, "segundos")



RMSE test: 1649.6260301421462
Tiempo de predicción: 0.6750562526285648 segundos


In [25]:
resultado_test = pd.DataFrame({
    'Modelo': ['LightGBM'],
    'Parámetros': [
        'n_estimators=400, learning_rate=0.05, random_state=54321'
    ],
    'RMSE test': [rmse_test],
    'Tiempo predicción (s)': [tiempo_pred_test]
})

resultado_test

,Modelo,Parámetros,RMSE test,Tiempo predicción (s)
0,LightGBM,"n_estimators=400, learning_rate=0.05, random_s...",1649.62603,0.675056


# Conclusiones

Conclusión Final
1. Resultados del Proyecto

El objetivo principal de este proyecto fue desarrollar un modelo de Machine Learning capaz de predecir con precisión y rapidez el valor de mercado de los autos usados para la nueva aplicación de Rusty Bargain. Tras un exhaustivo proceso de preparación de datos, ingeniería de características y optimización de hiperparámetros, se evaluaron cuatro arquitecturas distintas: Regresión Lineal, Random Forest, XGBoost y LightGBM.
Los modelos se analizaron bajo los tres criterios críticos para el negocio: calidad de predicción (RMSE), tiempo de entrenamiento y velocidad de predicción.

2. Justificación del Modelo Seleccionado

El modelo elegido para producción es LightGBM con los hiperparámetros n_estimators=400 y learning_rate=0.05. Los argumentos técnicos y de negocio que respaldan esta decisión son:

* Calidad de la predicción superior: Obtuvo el menor error en la etapa de validación (RMSE: 1642.75), superando consistentemente a sus competidores directos.

* Eficiencia en el entrenamiento: Registró un tiempo de entrenamiento de solo 10.21 segundos. En comparación, el modelo XGBoost con rendimiento similar requirió 946.35 segundos (más de 15 minutos). Esta velocidad de LightGBM facilita procesos futuros de reentrenamiento constante con nuevos datos del mercado automotriz.

* Velocidad de respuesta óptima: El tiempo de predicción de 0.70 segundos garantiza una experiencia de usuario fluida e instantánea dentro de la aplicación móvil de Rusty Bargain.

3. Desempeño en el Conjunto de Prueba (Test)

Para validar la robustez del modelo seleccionado antes de su despliegue, se realizaron pruebas con un conjunto de datos completamente independiente (test), obteniendo los siguientes resultados:

* RMSE Test: 1649.62603

* Tiempo de predicción: 0.675056 segundos

**Análisis del Test:**

* Ausencia de sobreajuste (Overfitting): La diferencia entre el RMSE de validación (1642.75) y el de prueba (1649.62) es de apenas un 0.4%. Esto demuestra que el modelo generaliza de forma excelente ante datos nuevos y reales.

* Consistencia técnica: El tiempo de predicción se redujo ligeramente en la prueba (0.67 s), confirmando que la aplicación responderá en menos de un segundo a las solicitudes de los clientes.

4. Impacto en el Negocio

* Retención de usuarios: Al ofrecer predicciones precisas y estables (bajo RMSE), los clientes recibirán valoraciones justas y realistas por sus vehículos, construyendo confianza en la marca.

* Escalabilidad del sistema: El bajísimo consumo de tiempo tanto en entrenamiento como en predicción reduce la carga computacional en los servidores de Rusty Bargain, permitiendo procesar miles de consultas simultáneas a un costo operativo mínimo.

**Recomendación final: Se aprueba el despliegue del modelo LightGBM (n_estimators=400, learning_rate=0.05) para su integración inmediata en la infraestructura productiva de la aplicación.**